In [2]:
# -*- coding: utf-8 -*-
import sqlite3
import requests
import time
from bs4 import BeautifulSoup

# DBファイルの保存先パス（相対パスで指定）
path = ''
db_name = 'test.db'

BASE = 'https://github.com'
ORG = 'google'
LIST_URL = f'{BASE}/{ORG}?tab=repositories'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; RepoScraper/1.0; +https://example.com)'
}

def parse_star_count(text):
    t = text.strip().lower().replace(',', '')
    try:
        if t.endswith('k'):
            return int(float(t[:-1]) * 1000)
        if t.endswith('m'):
            return int(float(t[:-1]) * 1000000)
        return int(t)
    except Exception as e:
        print("エラーが発生しました:", e)
        print("スター数の解析に失敗したため、0を返します。")
        return 0

def fetch_repo_list_page(page=1):
    url = LIST_URL + (f'&page={page}' if page > 1 else '')
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
    except Exception as e:
        print("エラーが発生しました:", e)
        print("ページ取得に失敗しました。Noneを返します。")
        return None
    finally:
        # リクエスト後は必ずスリープ
        time.sleep(1)

    if resp.status_code != 200:
        print("エラーが発生しました: HTTP", resp.status_code)
        print("ページ取得に失敗しました。Noneを返します。")
        return None

    return resp.text

def parse_repos_from_html(html):
    try:
        soup = BeautifulSoup(html, 'html.parser')
        items = soup.select("div[data-test-selector='repos-container'] li") \
            or soup.select("li[itemprop='owns']") \
            or soup.select("article")
        if not items:
            items = soup.select("li.col-12.d-flex.width-full.py-4.border-bottom.color-border-muted.public.source")

        repos = []
        for it in items:
            try:
                name_el = it.select_one("a[itemprop='name codeRepository'], h3 a, a[href^='/" + ORG + "/']")
                if not name_el:
                    continue
                name = name_el.get_text(strip=True).replace('\n', '').replace(' ', '')

                lang_el = it.select_one("[itemprop='programmingLanguage'], span[itemprop='programmingLanguage'], span.ml-0.mr-3 span")
                language = lang_el.get_text(strip=True) if lang_el else ''

                star_el = it.select_one("a[href$='/stargazers']")
                stars = parse_star_count(star_el.get_text(strip=True)) if star_el else 0

                repos.append((name, language, stars))
            except Exception as e:
                print("エラーが発生しました:", e)
                print("1件のリポジトリ解析に失敗しました。処理を継続します。")
                continue

        return repos
    except Exception as e:
        print("エラーが発生しました:", e)
        print("HTML解析に失敗しました。空リストを返します。")
        return []

def has_next_page(html):
    try:
        soup = BeautifulSoup(html, 'html.parser')
        next_btn = soup.select_one('a.next_page')
        if next_btn and 'disabled' not in (next_btn.get('class') or []):
            return True
        newer = soup.select_one("a[rel='next']")
        return newer is not None
    except Exception as e:
        print("エラーが発生しました:", e)
        print("ページネーション判定に失敗しました。次ページはないものとして扱います。")
        return False

# 1) DB接続（存在しなければ作成）
try:
    conn = sqlite3.connect(path + db_name)
except Exception as e:
    conn = None
    print("エラーが発生しました:", e)
    print("DB接続に失敗しました。続行できません。")
else:
    print("エラーは発生しませんでした。DB接続を確立しました。")
finally:
    if conn is not None:
        conn.close()
        print("DB接続を閉じました。")

# 2) テーブル作成
try:
    conn = sqlite3.connect(path + db_name)
    cur = conn.cursor()
    sql = 'CREATE TABLE IF NOT EXISTS repos (name TEXT PRIMARY KEY, language TEXT, stars INTEGER);'
    cur.execute(sql)
    conn.commit()
except Exception as e:
    print("エラーが発生しました:", e)
    print("テーブル作成に失敗しました。続行できません。")
finally:
    try:
        conn.close()
    except:
        pass

# 3) スクレイピングして挿入/更新
try:
    all_rows = []
    page = 1
    while True:
        html = fetch_repo_list_page(page)
        if html is None:
            print("ページ取得に失敗したため、スクレイピングを終了します。")
            break

        rows = parse_repos_from_html(html)
        all_rows.extend(rows)

        if not has_next_page(html):
            print("最終ページに到達しました。")
            break
        page += 1

    conn = sqlite3.connect(path + db_name)
    cur = conn.cursor()

    # 既存名の取得
    try:
        cur.execute("SELECT name FROM repos;")
        existing = {row[0] for row in cur.fetchall()}
    except Exception as e:
        existing = set()
        print("エラーが発生しました:", e)
        print("既存データの取得に失敗しました。空集合として処理します。")

    insert_sql = "INSERT INTO repos (name, language, stars) VALUES (?, ?, ?);"
    update_sql = "UPDATE repos SET language = ?, stars = ? WHERE name = ?;"

    to_insert = []
    to_update = []
    for name, lang, stars in all_rows:
        if name in existing:
            to_update.append((lang, stars, name))
        else:
            to_insert.append((name, lang, stars))

    try:
        if to_insert:
            cur.executemany(insert_sql, to_insert)
        if to_update:
            cur.executemany(update_sql, to_update)
        conn.commit()
    except Exception as e:
        print("エラーが発生しました:", e)
        print("データの挿入/更新に失敗しました。ロールバックは不要な変更のみです。")

except Exception as e:
    print("エラーが発生しました:", e)
    print("スクレイピング処理でエラーが発生しました。")

else:
    print("エラーは発生しませんでした。スクレイピングとDB反映が完了しました。")

finally:
    try:
        conn.close()
        print("DB接続を閉じました。")
    except:
        pass

# 4) SELECTで表示（スター数降順）
try:
    conn = sqlite3.connect(path + db_name)
    cur = conn.cursor()
    sql = "SELECT name, language, stars FROM repos ORDER BY stars DESC;"
    cur.execute(sql)
except Exception as e:
    print("エラーが発生しました:", e)
    print("SELECTの実行に失敗しました。")
else:
    print("エラーは発生しませんでした。SELECT結果を表示します。")
    for row in cur:
        name, language, stars = row
        print(name, language, stars)
finally:
    try:
        conn.close()
        print("DB接続を閉じました。")
    except:
        pass

エラーは発生しませんでした。DB接続を確立しました。
DB接続を閉じました。
最終ページに到達しました。
エラーは発生しませんでした。スクレイピングとDB反映が完了しました。
DB接続を閉じました。
エラーは発生しませんでした。SELECT結果を表示します。
DB接続を閉じました。
